# Object detection — Faster R-CNN with Torchvision

**DeepVision Masterclass — Modern Edition**

This notebook is designed as an executable teaching lab. Read the dataset contract and validation notes before training.

## 1. Detection task and annotations
Object detection predicts boxes and class labels. This tutorial uses a pretrained COCO detector for inference and defines the dataset contract for fine-tuning. Do not train on test images.

In [ ]:
import torch, torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
print(torch.__version__,torchvision.__version__)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Pretrained inference
The weights object provides the correct image preprocessing and category names.

In [ ]:
weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model=fasterrcnn_resnet50_fpn(weights=weights).to(device).eval()
preprocess=weights.transforms()
print('COCO classes:',len(weights.meta['categories']))
# Example with an image tensor in [0,1], shape [3,H,W]:
# image=preprocess(PIL_image).to(device)
# with torch.inference_mode(): prediction=model([image])[0]

## 3. Visualize predictions
Filter by confidence and draw boxes. Keep the original image dimensions when mapping coordinates.

In [ ]:
def filter_detections(prediction, threshold=.5):
    keep=prediction['scores']>=threshold
    return {k:v[keep].detach().cpu() for k,v in prediction.items()}
# Example: detections=filter_detections(prediction,.6)
# boxes=detections['boxes']; labels=detections['labels']; scores=detections['scores']

## 4. Custom dataset contract
A detection dataset returns `(image, target)` where target contains `boxes` (N×4, XYXY), `labels` (N, integer; background is 0 and object IDs start at 1), and optionally `image_id`, `area`, `iscrowd`.

In [ ]:
from torch.utils.data import Dataset, DataLoader
class DetectionDataset(Dataset):
    def __init__(self, images, annotations, transforms=None):
        self.images,self.annotations,self.transforms=images,annotations,transforms
    def __len__(self): return len(self.images)
    def __getitem__(self,idx):
        image=self.images[idx]  # load RGB image as float tensor [3,H,W] in [0,1]
        ann=self.annotations[idx]
        target={'boxes':torch.as_tensor(ann['boxes'],dtype=torch.float32).reshape(-1,4),
                'labels':torch.as_tensor(ann['labels'],dtype=torch.int64),
                'image_id':torch.tensor([idx])}
        if self.transforms: image,target=self.transforms(image,target)
        return image,target
def detection_collate(batch): return tuple(zip(*batch))

## 5. Replace the classifier head for custom classes
For K object categories, `num_classes=K+1` includes background. Use a dataset-specific label map and verify boxes visually before training.

In [ ]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
num_classes=3  # background + 2 object classes: replace with your label map
custom=fasterrcnn_resnet50_fpn(weights='DEFAULT')
in_features=custom.roi_heads.box_predictor.cls_score.in_features
custom.roi_heads.box_predictor=FastRCNNPredictor(in_features,num_classes)
custom.to(device)
# Optimizer example:
# params=[p for p in custom.parameters() if p.requires_grad]
# optimizer=torch.optim.SGD(params,lr=.005,momentum=.9,weight_decay=.0005)

## 6. Evaluation and pitfalls
Use COCO mAP (IoU 0.50:0.95), per-class AP, precision/recall and inference latency. Check coordinate conventions, empty targets, class IDs, train/val leakage and small-object performance.

## Exercises
Convert VOC XML or YOLO TXT annotations to XYXY; add horizontal-flip transforms that update boxes; compare Faster R-CNN with RetinaNet/SSD/YOLO; report mAP and latency.